# Evaluation — English ArXiv

Full evaluation with traditional metrics, abstraction metrics,
and LLM-as-Judge scoring.

**Memory strategy**: SigExt runs on CPU → unloaded → single LLM for both summary + judge.

In [1]:
!uv pip install -e ../..

Using Python 3.12.11 environment at: /home/zeus/miniconda3/envs/cloudspace
Resolved 133 packages in 1.00s                                       
   Building sm-sip @ file:///teamspace/studios/this_studio/sm-sip      
   Building sm-sip @ file:///teamspace/studios/this_studio/sm-sip1A
      Built sm-sip @ file:///teamspace/studios/this_studio/sm-sip1A
Prepared 1 package in 333ms                                              
Uninstalled 1 package in 1ms
Installed 1 package in 3ms file:///teamspace/studios/this_st
 ~ sm-sip==1.0.0 (from file:///teamspace/studios/this_studio/sm-sip)


In [2]:
from huggingface_hub import login
login()

## Configuration

In [3]:
from sm_sip.config import SigExtConfig, InferenceConfig, EvalConfig

sigext_config = SigExtConfig.from_preset("en", "1k-60t")
inference_config = InferenceConfig(lang="en", quantization="8bit", prompt_type="source_aware")
eval_config = EvalConfig(lang="en", judge_model_id="Qwen/Qwen2.5-14B-Instruct")

## Step 1: Load Data & Preprocess with SigExt (CPU)

In [4]:
from sm_sip.data import get_test_data
from sm_sip.models import load_sigext_model, unload_sigext_model, preprocess_dataset

test_data = get_test_data(lang="en", num_samples=100, skip_samples=sigext_config.skip_samples)

# SigExt runs on CPU to save GPU VRAM for the LLM
sigext_model, sigext_tokenizer = load_sigext_model(sigext_config.model_id, device="cpu")
processed_data = preprocess_dataset(test_data, sigext_model, sigext_tokenizer, lang="en")

# Free memory before loading LLM
unload_sigext_model(sigext_model, sigext_tokenizer)
print(f"Preprocessed {len(processed_data)} samples.")

  Loading EN test data (skipping 0)...
  Loaded 100 ArXiv samples (skipped 0)
  Loading SigExt model on cpu: LookUpMark/sigext-arxiv-en-1k-060t...


Loading weights:   0%|          | 0/535 [00:00<?, ?it/s]

  Extracting salient sentences:   0%|          | 0/100 [00:00<?, ?it/s]

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.8/12.8 MB 123.0 MB/s  0:00:00
✔ Download and installation successful
You can now load the package via spacy.load('en_core_web_sm')
⚠ Restart to reload dependencies
If you are in a Jupyter or Colab notebook, you may need to restart Python in
order to load all the package's dependencies. You can do this by selecting the
'Restart kernel' or 'Restart runtime' option.



[notice] A new release of pip is available: 26.0 -> 26.0.1
[notice] To update, run: pip install --upgrade pip
Input ids are automatically padded to be a multiple of `config.attention_window`: 512


  SigExt model unloaded.
Preprocessed 100 samples.


## Step 2: Load Single LLM (Summary + Judge)

In [5]:
from sm_sip.models import load_llm, create_summary_chain, create_judge_chain
from sm_sip.prompts import get_summary_prompt, get_judge_prompt

# Single LLM for both tasks
llm_model, llm_tokenizer, gen_pipe = load_llm(
    inference_config.llm_model_id,
    inference_config.quantization,
    seed=inference_config.seed,
)

# Two chains, same underlying model
summary_chain = create_summary_chain(gen_pipe, get_summary_prompt("en", "source_aware"))
judge_chain = create_judge_chain(gen_pipe, get_judge_prompt("unified"))

  Loading LLM (8bit, seed=42): meta-llama/Llama-3.1-8B-Instruct...


Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/184 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/55.4k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/9.09M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/296 [00:00<?, ?B/s]

The following generation flags are not valid and may be ignored: ['top_p']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
Passing `generation_config` together with generation-related arguments=({'max_new_tokens', 'do_sample', 'temperature'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.


## Step 3: Run Enhanced Evaluation

In [6]:
from sm_sip.pipelines import run_enhanced_evaluation

metrics, samples = run_enhanced_evaluation(
    processed_data,
    summary_chain,
    judge_chain=judge_chain,
    lang="en",
)

print("\n=== RESULTS ===")
for key, val in metrics.items():
    print(f"  {key}: {val['mean']:.4f} ± {val['std']:.4f}")

    Generating & Evaluating:   0%|          | 0/100 [00:00<?, ?it/s]

The following generation flags are not valid and may be ignored: ['temperature', 'top_p']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
Both `max_new_tokens` (=512) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
/home/zeus/miniconda3/envs/cloudspace/lib/python3.12/site-packages/bitsandbytes/autograd/_functions.py:123: UserWarning: MatMul8bitLt: inputs will be cast from torch.bfloat16 to float16 during quantization
  warnings.warn(f"MatMul8bitLt: inputs will be cast from {A.dtype} to float16 during quantization")
Both `max_new_tokens` (=512) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


    Error: CUDA out of memory. Tried to allocate 1.71 GiB. GPU 0 has a total capacity of 14.74 GiB of which 496.19 MiB is free. Process 234251 has 14.25 GiB memory in use. Of the allocated memory 10.62 GiB is allocated by PyTorch, and 3.46 GiB is reserved by PyTorch but unallocated. If reserved but unallocated memory is large try setting PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True to avoid fragmentation.  See documentation for Memory Management  (https://pytorch.org/docs/stable/notes/cuda.html#environment-variables)


/home/zeus/miniconda3/envs/cloudspace/lib/python3.12/site-packages/bitsandbytes/autograd/_functions.py:123: UserWarning: MatMul8bitLt: inputs will be cast from torch.bfloat16 to float16 during quantization
  warnings.warn(f"MatMul8bitLt: inputs will be cast from {A.dtype} to float16 during quantization")


config.json:   0%|          | 0.00/482 [00:00<?, ?B/s]

...

Loading weights:   0%|          | 0/389 [00:00<?, ?it/s]

RobertaModel LOAD REPORT from: roberta-large
Key                             | Status     | 
--------------------------------+------------+-
lm_head.bias                    | UNEXPECTED | 
lm_head.dense.bias              | UNEXPECTED | 
lm_head.layer_norm.bias         | UNEXPECTED | 
roberta.embeddings.position_ids | UNEXPECTED | 
lm_head.layer_norm.weight       | UNEXPECTED | 
lm_head.dense.weight            | UNEXPECTED | 
pooler.dense.bias               | MISSING    | 
pooler.dense.weight             | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.
Both `max_new_tokens` (=512) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_cla

Loading weights:   0%|          | 0/389 [00:00<?, ?it/s]

RobertaModel LOAD REPORT from: roberta-large
Key                             | Status     | 
--------------------------------+------------+-
lm_head.bias                    | UNEXPECTED | 
lm_head.dense.bias              | UNEXPECTED | 
lm_head.layer_norm.bias         | UNEXPECTED | 
roberta.embeddings.position_ids | UNEXPECTED | 
lm_head.layer_norm.weight       | UNEXPECTED | 
lm_head.dense.weight            | UNEXPECTED | 
pooler.dense.bias               | MISSING    | 
pooler.dense.weight             | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.
Both `max_new_tokens` (=512) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_cla


=== RESULTS ===
  bert: 0.8357 ± 0.0179
  rouge1: 0.4010 ± 0.0825
  rougeL: 0.2191 ± 0.0621
  kir: 0.6680 ± 0.2874
  abstraction: 0.6445 ± 0.1053
  compression: 0.1246 ± 0.0835
  novel_ngrams: 0.4597 ± 0.0880
  judge_faithfulness: 4.7800 ± 0.6096
  judge_completeness: 4.6600 ± 0.6515
  judge_conciseness: 4.7400 ± 0.6264
  judge_abstraction: 4.7600 ± 0.6499


## Step 4: Save Results

In [7]:
from sm_sip.utils.io import save_results
from datetime import datetime

save_results({
    "run_info": {
        "timestamp": datetime.now().isoformat(),
        "sigext_model": sigext_config.model_id,
        "llm_model": inference_config.llm_model_id,
        "quantization": inference_config.quantization,
        "prompt_type": inference_config.prompt_type,
        "seed": inference_config.seed,
        "num_samples": len(samples),
    },
    "metrics": metrics,
    "samples": samples,
}, "results/english/eval_enhanced.json")

print("Results saved!")

  Results saved: results/english/eval_enhanced.json
Results saved!


## Cleanup

In [8]:
from sm_sip.utils.gpu import clear_gpu_memory
del llm_model, llm_tokenizer, gen_pipe
clear_gpu_memory()